# M3GLVQ Cache-Analyse — Top-20, Gewichte und dominante Views

Dieses Notebook startet bewusst deskriptiv:

1. Cache laden und Vollständigkeit prüfen
2. Top-20 vollständige Konfigurationen je Methode
3. globale View-Gewichte analysieren
4. label-spezifische View-Gewichte analysieren
5. dominante Views je Fold und Label auswerten
6. η-Abdeckung prüfen, bevor η-Werte miteinander verglichen werden

**View-Reihenfolge:** `naics`, `hs`, `am`.

In [2]:
import sqlite3, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)

# Pfad bei Bedarf anpassen
CACHE_DB = Path("outputs/cache_0201.sqlite")
EXPECTED_FOLDS = 10
TOP_N = 20
VIEWS = ["naics", "hs", "am"]

assert CACHE_DB.exists(), f"Cache-Datei nicht gefunden: {CACHE_DB.resolve()}"

In [3]:
con = sqlite3.connect(str(CACHE_DB))

rows = pd.read_sql_query("SELECT * FROM fold_results", con)

print("Fold-Zeilen gesamt:", len(rows))
display(
    rows.groupby(["model", "status"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values(["model", "status"])
)

print("η-Werte:", sorted(rows["eta"].dropna().unique()))
print("K-Werte:", sorted(rows["k_value"].dropna().astype(str).unique())[:20], "...")

Fold-Zeilen gesamt: 8026


,model,status,n
0,C0_primary,ok,5
1,C1_primary,ok,7
2,M3GLVQ_Global,ok,3910
3,M3GLVQ_Label,ok,3907
4,am_only,ok,10
5,hs_only,ok,10
6,naics_only,ok,10
7,svglobal::am,degenerate,7
8,svglobal::am,ok,20
9,svglobal::hs,degenerate,5


η-Werte: [0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05]
K-Werte: ['10/10', '10/3', '10/4', '10/5', '10/6', '10/7', '10/8', '10/9', '3/10', '3/3', '3/4', '3/5', '3/6', '3/7', '3/8', '3/9', '4/10', '4/3', '4/4', '4/5'] ...


## 1. Konfigurationen auf OOF-Ebene aggregieren

Für das Ranking werden nur Konfigurationen verwendet, bei denen alle erwarteten Folds den Status `ok` besitzen.
Die Vorhersagen aller Folds werden anschließend zu einem vollständigen OOF-Vektor zusammengefügt.

In [5]:
def aggregate_config(g):
    ok = g[g["status"] == "ok"].sort_values("fold")

    out = {
        "n_rows": len(g),
        "n_ok": len(ok),
    }

    if len(ok) == 0:
        for key in ["bal_acc", "acc", "recall0", "recall1", "f1_macro", "mcc"]:
            out[key] = np.nan
        return pd.Series(out)

    yt = np.concatenate([
        np.asarray(json.loads(s), dtype=int)
        for s in ok["y_true_json"]
    ])
    yp = np.concatenate([
        np.asarray(json.loads(s), dtype=int)
        for s in ok["y_pred_json"]
    ])

    out.update({
        "bal_acc": balanced_accuracy_score(yt, yp),
        "acc": accuracy_score(yt, yp),
        "recall0": recall_score(yt, yp, pos_label=0, zero_division=0),
        "recall1": recall_score(yt, yp, pos_label=1, zero_division=0),
        "f1_macro": f1_score(yt, yp, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(yt, yp),
    })
    return pd.Series(out)

summary = (
    rows.groupby(["config_key", "model", "k_value", "eta"], dropna=False)
        .apply(aggregate_config)
        .reset_index()
)

complete = summary[summary["n_ok"] == EXPECTED_FOLDS].copy()
partial = summary[(summary["n_ok"] > 0) & (summary["n_ok"] < EXPECTED_FOLDS)].copy()

coverage = (
    summary.groupby("model")
        .agg(
            configs=("config_key", "count"),
            complete10=("n_ok", lambda s: int((s == EXPECTED_FOLDS).sum())),
            partial=("n_ok", lambda s: int(((s > 0) & (s < EXPECTED_FOLDS)).sum())),
        )
        .reset_index()
)
display(coverage)

,model,configs,complete10,partial
0,C0_primary,1,0,1
1,C1_primary,1,0,1
2,M3GLVQ_Global,391,391,0
3,M3GLVQ_Label,391,390,1
4,am_only,1,1,0
5,hs_only,1,1,0
6,naics_only,1,1,0
7,svglobal::am,8,1,3
8,svglobal::hs,8,3,5
9,svglobal::naics,8,8,0


## 2. Top-20 je Methode

`Global` und `Label` besitzen genügend Konfigurationen für echte Top-20-Listen.
Bei Single Views werden entsprechend nur die vorhandenen vollständigen K-Konfigurationen ausgegeben.

In [6]:
RANK_COLS = [
    "k_value", "eta", "bal_acc", "acc",
    "recall0", "recall1", "f1_macro", "mcc", "n_ok"
]

def top_configs(model, n=TOP_N):
    d = (
        complete[complete["model"] == model]
        .sort_values(["bal_acc", "mcc"], ascending=False)
        .head(n)
        .copy()
    )
    return d

for model in [
    "M3GLVQ_Global",
    "M3GLVQ_Label",
    "svglobal::naics",
    "svglobal::hs",
    "svglobal::am",
    "uniform",
]:
    d = top_configs(model)
    if len(d):
        print(f"\n=== {model}: Top {min(n:=TOP_N, len(d))} ===")
        display(d[RANK_COLS].reset_index(drop=True))


=== M3GLVQ_Global: Top 20 ===


,k_value,eta,bal_acc,acc,recall0,recall1,f1_macro,mcc,n_ok
0,3/3,0.015,0.585540,0.589136,0.442756,0.728324,0.578680,0.178698,10.0
1,5/5,0.010,0.577773,0.579259,0.518744,0.636802,0.576969,0.156666,10.0
2,3/4,0.010,0.576778,0.580741,0.419453,0.734104,0.567982,0.161943,10.0
3,6/6,0.050,0.573112,0.573333,0.564336,0.581888,0.573103,0.146211,10.0
4,3/3,0.020,0.571313,0.574321,0.451874,0.690751,0.566558,0.146966,10.0
5,4/4,0.005,0.570956,0.573827,0.456940,0.684971,0.566684,0.145831,10.0
6,3/4,0.050,0.570266,0.572840,0.468085,0.672447,0.566958,0.143626,10.0
7,3/3,0.010,0.570126,0.573333,0.442756,0.697495,0.564587,0.145130,10.0
8,4/6,0.020,0.569651,0.571852,0.482270,0.657033,0.567375,0.141525,10.0
9,3/8,0.050,0.569345,0.572840,0.430598,0.708092,0.562589,0.144467,10.0



=== M3GLVQ_Label: Top 20 ===


,k_value,eta,bal_acc,acc,recall0,recall1,f1_macro,mcc,n_ok
0,4/3,0.005,0.571287,0.572840,0.509625,0.632948,0.570355,0.143690,10.0
1,3/3,0.005,0.568341,0.572346,0.409321,0.727360,0.559099,0.144298,10.0
2,5/3,0.005,0.561626,0.561481,0.567376,0.555877,0.561451,0.123218,10.0
3,5/4,0.005,0.557641,0.558519,0.522796,0.592486,0.557461,0.115550,10.0
4,5/6,0.005,0.557068,0.558519,0.499493,0.614644,0.556243,0.114904,10.0
5,3/6,0.005,0.555484,0.559012,0.415400,0.695568,0.548289,0.115660,10.0
6,6/6,0.005,0.553629,0.554074,0.535968,0.571291,0.553628,0.107302,10.0
7,4/4,0.005,0.551579,0.553580,0.472138,0.631021,0.549657,0.104498,10.0
8,8/6,0.005,0.551022,0.551605,0.527862,0.574181,0.550989,0.102133,10.0
9,10/6,0.010,0.550730,0.550617,0.555218,0.546243,0.550578,0.101431,10.0



=== svglobal::naics: Top 8 ===


...

,k_value,eta,bal_acc,acc,recall0,recall1,f1_macro,mcc,n_ok
0,4/4,0.0,0.570979,0.572346,0.516717,0.625241,0.570321,0.142813,10.0
1,5/5,0.0,0.567849,0.569877,0.487335,0.648362,0.565975,0.137527,10.0
2,3/3,0.0,0.567757,0.568889,0.522796,0.612717,0.567370,0.136064,10.0



=== svglobal::am: Top 1 ===


,k_value,eta,bal_acc,acc,recall0,recall1,f1_macro,mcc,n_ok
0,5/5,0.0,0.544738,0.548148,0.409321,0.680154,0.53787,0.092981,10.0



=== uniform: Top 1 ===


,k_value,eta,bal_acc,acc,recall0,recall1,f1_macro,mcc,n_ok
0,5/5,0.0,0.564436,0.566914,0.466059,0.662813,0.561353,0.131488,10.0


## 3. Gewichte in Long-Format bringen

Die gespeicherten `a_sq`-Werte werden direkt als finale View-Gewichte interpretiert.

- `Global`: ein Gewichtsvektor pro Fold
- `Label`: ein Gewichtsvektor pro Fold und Label

In [ ]:
def parse_weight_rows(df):
    records = []

    for _, r in df[(df["status"] == "ok") & df["vweights_json"].notna()].iterrows():
        w = json.loads(r["vweights_json"])

        if w["kind"] == "global":
            arr = np.asarray(w["a_sq"], dtype=float)
            for view, weight in zip(VIEWS, arr):
                records.append({
                    "config_key": r["config_key"],
                    "model": r["model"],
                    "k_value": r["k_value"],
                    "eta": r["eta"],
                    "fold": r["fold"],
                    "label": "global",
                    "view": view,
                    "weight": weight,
                })

        elif w["kind"] == "label":
            labels = w["labels"]
            arr = np.asarray(w["a_sq"], dtype=float)

            for i, label in enumerate(labels):
                for view, weight in zip(VIEWS, arr[i]):
                    records.append({
                        "config_key": r["config_key"],
                        "model": r["model"],
                        "k_value": r["k_value"],
                        "eta": r["eta"],
                        "fold": r["fold"],
                        "label": str(label),
                        "view": view,
                        "weight": weight,
                    })

    return pd.DataFrame(records)

weights = parse_weight_rows(rows)
weights.head()

In [ ]:
def fold_weight_table(model):
    d = weights[weights["model"] == model].copy()

    p = (
        d.pivot_table(
            index=["config_key", "k_value", "eta", "fold", "label"],
            columns="view",
            values="weight",
        )
        .reset_index()
    )

    p["dominant_view"] = p[VIEWS].idxmax(axis=1)
    p["max_weight"] = p[VIEWS].max(axis=1)

    arr = p[VIEWS].to_numpy()
    p["entropy"] = -(arr * np.log(arr + 1e-12)).sum(axis=1) / np.log(len(VIEWS))
    return p

global_fold_weights = fold_weight_table("M3GLVQ_Global")
label_fold_weights  = fold_weight_table("M3GLVQ_Label")

## 4. Global — Top-20 und gelernte View-Gewichte

Neben der Performance werden pro Konfiguration ausgegeben:

- mittleres Gewicht je View über die 10 Folds
- mittlere normierte Entropie der Gewichte
- Anzahl der Folds, in denen NAICS / HS / AM dominant war

In [ ]:
def summarize_global_weights(config_keys):
    d = global_fold_weights[global_fold_weights["config_key"].isin(config_keys)].copy()

    means = (
        d.groupby(["config_key", "k_value", "eta"])
         .agg(
             naics_mean=("naics", "mean"),
             hs_mean=("hs", "mean"),
             am_mean=("am", "mean"),
             entropy_mean=("entropy", "mean"),
             max_weight_mean=("max_weight", "mean"),
         )
         .reset_index()
    )

    dom = (
        d.groupby(["config_key", "k_value", "eta", "dominant_view"])
         .size()
         .unstack(fill_value=0)
         .reset_index()
    )

    for v in VIEWS:
        if v not in dom.columns:
            dom[v] = 0

    return means.merge(
        dom[["config_key", "k_value", "eta"] + VIEWS],
        on=["config_key", "k_value", "eta"],
        how="left",
    )

global_top20 = top_configs("M3GLVQ_Global")
global_top20_w = global_top20.merge(
    summarize_global_weights(global_top20["config_key"]),
    on=["config_key", "k_value", "eta"],
    how="left",
)

display(
    global_top20_w[[
        "k_value", "eta", "bal_acc", "mcc",
        "naics_mean", "hs_mean", "am_mean",
        "entropy_mean",
        "naics", "hs", "am",
    ]].reset_index(drop=True)
)

In [ ]:
# Dominante Views über alle Fold-Konfigurationen der Global-Top-20
g = global_fold_weights[
    global_fold_weights["config_key"].isin(global_top20["config_key"])
]

display(
    (g["dominant_view"].value_counts(normalize=True) * 100)
    .rename("percent")
    .round(1)
    .to_frame()
)

display(
    g[VIEWS + ["entropy", "max_weight"]]
    .mean()
    .rename("Top20 mean")
    .to_frame()
)

## 5. Label-spezifisch — Top-20, getrennt für Label 0 und Label 1

Hier ist entscheidend, nicht nur die Mittelwerte anzusehen.
Zusätzlich wird gezählt, welche View pro Fold und Label dominant war.

In [ ]:
def summarize_label_weights(config_keys):
    d = label_fold_weights[label_fold_weights["config_key"].isin(config_keys)].copy()

    means = (
        d.groupby(["config_key", "k_value", "eta", "label"])
         .agg(
             naics_mean=("naics", "mean"),
             hs_mean=("hs", "mean"),
             am_mean=("am", "mean"),
             entropy_mean=("entropy", "mean"),
             max_weight_mean=("max_weight", "mean"),
         )
         .reset_index()
    )

    dom = (
        d.groupby(["config_key", "k_value", "eta", "label", "dominant_view"])
         .size()
         .unstack(fill_value=0)
         .reset_index()
    )

    for v in VIEWS:
        if v not in dom.columns:
            dom[v] = 0

    return means.merge(
        dom[["config_key", "k_value", "eta", "label"] + VIEWS],
        on=["config_key", "k_value", "eta", "label"],
        how="left",
    )

label_top20 = top_configs("M3GLVQ_Label")
label_top20_w = summarize_label_weights(label_top20["config_key"])

for lab in ["0", "1"]:
    print(f"\n=== Label {lab} ===")
    x = label_top20.merge(
        label_top20_w[label_top20_w["label"] == lab],
        on=["config_key", "k_value", "eta"],
        how="left",
    )
    display(
        x[[
            "k_value", "eta", "bal_acc",
            "naics_mean", "hs_mean", "am_mean",
            "entropy_mean",
            "naics", "hs", "am",
        ]].reset_index(drop=True)
    )

In [ ]:
# Aggregierte dominante Views der Label-Top-20
l = label_fold_weights[
    label_fold_weights["config_key"].isin(label_top20["config_key"])
]

for lab in ["0", "1"]:
    q = l[l["label"] == lab]

    print(f"\nLabel {lab}: dominante Views")
    display(
        (q["dominant_view"].value_counts(normalize=True) * 100)
        .rename("percent")
        .round(1)
        .to_frame()
    )

    print(f"Label {lab}: mittlere Gewichte")
    display(
        q[VIEWS + ["entropy", "max_weight"]]
        .mean()
        .rename("mean")
        .to_frame()
    )

## 6. η-Abdeckung zuerst prüfen

Ein η-Mittelwert ist nur sinnvoll mit Blick auf die Anzahl berechneter K-Konfigurationen.
Insbesondere dürfen teilweise durchgerechnete η-Werte nicht so behandelt werden, als sei bereits das gesamte 8×8-K-Grid vorhanden.

In [ ]:
def eta_performance(model):
    d = complete[complete["model"] == model].copy()
    return (
        d.groupby("eta")
         .agg(
             n_configs=("config_key", "count"),
             mean_bal_acc=("bal_acc", "mean"),
             median_bal_acc=("bal_acc", "median"),
             best_bal_acc=("bal_acc", "max"),
             sd_bal_acc=("bal_acc", "std"),
         )
         .reset_index()
         .sort_values("eta")
    )

print("GLOBAL")
display(eta_performance("M3GLVQ_Global"))

print("LABEL")
display(eta_performance("M3GLVQ_Label"))

In [ ]:
# Einfache Visualisierung: Performance-Verteilung je η
for model in ["M3GLVQ_Global", "M3GLVQ_Label"]:
    d = complete[complete["model"] == model].copy()

    fig, ax = plt.subplots(figsize=(8, 4))
    grouped = [
        d.loc[d["eta"] == eta, "bal_acc"].to_numpy()
        for eta in sorted(d["eta"].dropna().unique())
    ]
    labels = [str(x) for x in sorted(d["eta"].dropna().unique())]

    ax.boxplot(grouped, tick_labels=labels)
    ax.set_title(f"{model}: Balanced Accuracy nach η")
    ax.set_xlabel("η")
    ax.set_ylabel("Balanced Accuracy")
    plt.show()

## 7. Nächste Analyseschritte

Nach dem Top-20-Screening:

1. Global vs. Label exakt nach `(K0, K1, η, Fold)` paaren
2. Gewichts-Sparsität gegen Performance analysieren
3. Label 0 / Label 1: dominante View gegen jeweiligen Klassen-Recall stellen
4. K0/K1-Heatmaps je η
5. Fold-Stabilität der Gewichte
6. Prototypen und Prototype-Collapse separat analysieren

In [ ]:
con.close()